In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))
import config

# Verify data is accessible
try:
    config.assert_data_exists()
    print("✓ Data path:", config.DATA_ROOT)
except FileNotFoundError as e:
    print("✗ Data path error:", e)

# Data loading utility — use this instead of pd.read_csv() directly
# It handles column name cleaning (the CSV has spaces in headers)
def load_csv(path):
    """Load a CSV from config.DATA_RAW or config.DATA_PROCESSED with cleaned column names."""
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    return df

# Example usage:
# df = load_csv(config.DATA_RAW / "Airlines.csv")
# df = load_csv(config.DATA_PROCESSED / "Airlines_enriched.csv")

# 04 — Hyperparameter Tuning
**CMPE 188 | Flight Delay Prediction**

Goals:
- Load the enriched dataset (from notebook 02)
- Run GridSearchCV on XGBoost — completes the TODO from `scripts/xgboost_pipeline.py`
- Run RandomizedSearchCV on Random Forest
- 5-fold stratified cross-validation throughout
- Compare tuned models against notebook 03 baselines

In [ ]:
import pandas as pd
from sklearn.model_selection import GridSearchCV, RandomizedSearchCV, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

DATA_PATH = str(config.DATA_PROCESSED / 'Airlines_enriched.csv')
df = pd.read_csv(DATA_PATH)
df.head()

## 1. Preprocessing + Train/Test Split

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, MinMaxScaler
from sklearn.feature_selection import SelectKBest, chi2
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score, classification_report

target = "Delay"
X = df.drop(columns=target)
y = df[target]

categorical_cols = ["Airline", "AirportFrom", "AirportTo", "time_bucket"]
numeric_cols = [
    "DayOfWeek", "Time", "Length", "is_peak_hour", "route_volume", "airline_delay_rate",
    # Enriched geographic + weather features
    "from_lat", "from_lon", "from_elevation_ft",
    "from_avg_temperature", "from_avg_precipitation", "from_avg_wind_speed",
    "to_lat", "to_lon", "to_elevation_ft",
    "to_avg_temperature", "to_avg_precipitation", "to_avg_wind_speed",
]

# Keep only columns that exist in the enriched dataset
categorical_cols = [col for col in categorical_cols if col in X.columns]
numeric_cols = [col for col in numeric_cols if col in X.columns]

preprocessor = ColumnTransformer(
    transformers=[
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), categorical_cols),
        ("num", MinMaxScaler(), numeric_cols),
    ]
)

selector = SelectKBest(score_func=chi2, k=50)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"Train: {X_train.shape}, Test: {X_test.shape}")

## 2. GridSearchCV — XGBoost

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", XGBClassifier(eval_metric="logloss", random_state=42)),
])

param_grid = {
    "classifier__n_estimators": [100, 200, 300],
    "classifier__max_depth": [3, 5, 7],
    "classifier__learning_rate": [0.01, 0.1, 0.2],
    "classifier__subsample": [0.8, 1.0],
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

xgb_grid = GridSearchCV(
    xgb_pipeline,
    param_grid,
    cv=cv,
    scoring="roc_auc",
    n_jobs=-1,
)

xgb_grid.fit(X_train, y_train)

print(f"Best Parameters: {xgb_grid.best_params_}")
print(f"Best ROC-AUC (CV): {xgb_grid.best_score_:.4f}")

## 3. RandomizedSearchCV — Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("selector", selector),
    ("classifier", RandomForestClassifier(random_state=42, n_jobs=-1)),
])

param_distributions = {
    "classifier__n_estimators": [100, 200, 300, 500],
    "classifier__max_depth": [5, 10, 15, 20, None],
    "classifier__min_samples_split": [2, 5, 10],
    "classifier__max_features": ["sqrt", "log2", 0.5],
}

rf_random = RandomizedSearchCV(
    rf_pipeline,
    param_distributions,
    n_iter=20,
    cv=StratifiedKFold(n_splits=5, shuffle=True, random_state=42),
    scoring="roc_auc",
    random_state=42,
    n_jobs=-1,
)

rf_random.fit(X_train, y_train)

print(f"Best Parameters: {rf_random.best_params_}")
print(f"Best ROC-AUC (CV): {rf_random.best_score_:.4f}")

## 4. Baseline vs Tuned Comparison Table

In [ ]:
# Evaluate tuned models on test set
xgb_pred = xgb_grid.predict(X_test)
xgb_proba = xgb_grid.predict_proba(X_test)[:, 1]
xgb_tuned_acc = accuracy_score(y_test, xgb_pred)
xgb_tuned_auc = roc_auc_score(y_test, xgb_proba)

rf_pred = rf_random.predict(X_test)
rf_proba = rf_random.predict_proba(X_test)[:, 1]
rf_tuned_acc = accuracy_score(y_test, rf_pred)
rf_tuned_auc = roc_auc_score(y_test, rf_proba)

results = pd.DataFrame({
    "Model": ["XGBoost baseline", "XGBoost tuned", "RF baseline", "RF tuned"],
    "Features": ["enriched + derived"] * 4,
    "ROC-AUC": [0.6895, xgb_tuned_auc, 0.6848, rf_tuned_auc],
    "Accuracy": [0.6438, xgb_tuned_acc, 0.6384, rf_tuned_acc],
})
print(results.to_string(index=False))